# Phase 2 Service Panel Volatility

This notebook inspects `service_panel_route_period.csv` and highlights route-direction-period combinations with the highest historical volatility.

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent

PROCESSED = ROOT / 'data' / 'processed'
manifest_path = PROCESSED / 'feeds_manifest.csv'
route_panel_path = PROCESSED / 'service_panel_route_period.csv'
stop_panel_path = PROCESSED / 'service_panel_stop_level.csv'

manifest = pd.read_csv(manifest_path, dtype=str)
route_panel = pd.read_csv(route_panel_path, dtype={'version_id': str, 'route_id': str, 'direction_id': str})
stop_panel = pd.read_csv(stop_panel_path, dtype={'version_id': str, 'stop_id': str})

print('Route panel rows:', len(route_panel))
print('Stop panel rows:', len(stop_panel))
print('Versions in panel:', route_panel['version_id'].nunique())
route_panel.head()

In [ ]:
route_panel = route_panel.copy()
route_panel['trip_count'] = pd.to_numeric(route_panel['trip_count'], errors='coerce')
route_panel['avg_headway'] = pd.to_numeric(route_panel['avg_headway'], errors='coerce')
route_panel['service_hours'] = pd.to_numeric(route_panel['service_hours'], errors='coerce')

version_dates = manifest[['version_id', 'date_start']].drop_duplicates()
version_dates['date_start'] = pd.to_datetime(version_dates['date_start'], errors='coerce')

route_panel = route_panel.merge(version_dates, on='version_id', how='left')
route_panel['series_key'] = (
    route_panel['route_id'].astype(str)
    + ' | dir=' + route_panel['direction_id'].astype(str)
    + ' | ' + route_panel['time_period'].astype(str)
)

print('Missing date_start rows:', route_panel['date_start'].isna().sum())
print('Unique route-direction-period series:', route_panel['series_key'].nunique())

In [ ]:
volatility = (
    route_panel.groupby('series_key', as_index=False)['trip_count']
    .std(ddof=0)
    .rename(columns={'trip_count': 'trip_count_std'})
    .sort_values('trip_count_std', ascending=False)
)

top_20 = volatility.head(20)
top_20

In [ ]:
top_keys = top_20['series_key'].head(8).tolist()
plot_df = route_panel[route_panel['series_key'].isin(top_keys)].copy()
plot_df = plot_df.sort_values('date_start')

plt.figure(figsize=(14, 8))
sns.lineplot(data=plot_df, x='date_start', y='trip_count', hue='series_key', linewidth=1.8)
plt.title('Top Volatile Route-Direction-Period Series by Trip Count')
plt.xlabel('Feed Start Date')
plt.ylabel('Trip Count')
plt.legend(title='Series', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
period_ts = (
    route_panel.groupby(['date_start', 'time_period'], as_index=False)['trip_count']
    .sum()
    .sort_values('date_start')
)

period_vol = (
    period_ts.groupby('time_period', as_index=False)['trip_count']
    .std(ddof=0)
    .rename(columns={'trip_count': 'std_total_trip_count'})
    .sort_values('std_total_trip_count', ascending=False)
)

display(period_vol)

plt.figure(figsize=(12, 6))
sns.lineplot(data=period_ts, x='date_start', y='trip_count', hue='time_period', linewidth=2)
plt.title('System-Wide Trip Count by Time Period Over Time')
plt.xlabel('Feed Start Date')
plt.ylabel('Total Trip Count')
plt.tight_layout()
plt.show()